# Batch Phase Fitting

The generic batch runner preserves one output row per input, including
failures. The smoke path demonstrates that alignment with a peak plan;
in real work replace the analyzer with `PhaseFitAnalyzer` and explicit
CIF-derived phase models before running the button.


In [ ]:
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display

from xrd_tools.analysis import AnalysisInput, PeakFitAnalyzer, PeakFitPlan, batch_params_table, run_batch
from xrd_tools.gui.widgets import BatchPhaseFitViewer, PhaseFitControls


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
q = np.linspace(2.3, 3.2, 260)
run_button = widgets.Button(description="Run batch fit", button_style="primary")
quality = widgets.FloatSlider(value=0.02, min=0.005, max=0.1, step=0.005, description="width", continuous_update=False)
status = widgets.HTML("<i>Smoke uses a bounded three-pattern batch.</i>")
output = widgets.Output()
display(widgets.VBox([quality, widgets.HBox([run_button]), status, output]))


In [ ]:
NOTEBOOK_STATE = {"runs": 0, "outcomes": []}

def run_batch_fit(_=None):
    with output:
        clear_output(wait=True)
        try:
            patterns = [8 + 80 * np.exp(-0.5 * ((q - center) / quality.value) ** 2) for center in (2.755, 2.760, 2.765)]
            analyzer = PeakFitAnalyzer(PeakFitPlan(positions=(2.76,), model="gaussian", background="linear", sigma_init=quality.value))
            outcomes = run_batch(analyzer, [AnalysisInput(str(i), q, y, x_unit="q_A^-1") for i, y in enumerate(patterns)])
            labels, columns = batch_params_table(outcomes)
            assert len(labels) == len(patterns) == len(columns["center_0"])
            display(columns)
            NOTEBOOK_STATE.update(runs=NOTEBOOK_STATE["runs"] + 1, outcomes=outcomes, columns=columns)
            status.value = f"<b>Fit {len(outcomes)} aligned patterns with width={quality.value:.3f}.</b>"
        except Exception as exc:
            status.value = f"<b>Batch fit failed:</b> {exc}"
            raise

run_button.on_click(run_batch_fit)
NOTEBOOK_ACTIONS = {"run_batch_fit": run_batch_fit}
if SMOKE_MODE or os.environ.get("XDART_NOTEBOOK_AUTORUN") == "1":
    run_batch_fit()
